# RealWorld HAR — Raw Signal Windowing for Deep Learning

This notebook loads the segment-aware cleaned dataset (`realworld_clean.pkl`)
and produces fixed-length raw-signal windows ready for deep learning models
(DNN, CNN1D, RNN, DeepConvLSTM, TinyHAR) — the same `X_windows` /
`y_windows` / `groups` format used in the earlier student-dataset
model-comparison notebook.

Unlike the feature-extraction notebook, no statistics are computed here —
each window keeps its full raw 9-channel time series
(`body_x/y/z`, `gyr_x/y/z`, `mag_x/y/z`).

**Same segment-safety rule applies:** gravity filtering, sensor
synchronization, and windowing all happen *within each continuous
segment* (`segment_id`), never across a recording gap.

Pipeline:
1. Load cleaned data
2. Pre-index gyr/mag by (participant, activity, position) for fast lookup
3. Per segment: gravity filter → synchronize sensors → slide raw windows
4. Stack into `X_windows`, encode `y_windows` and `groups`
5. Save as `.npy` files

In [1]:
import time
import numpy as np
import pandas as pd

from scipy.signal import butter, filtfilt
from sklearn.preprocessing import LabelEncoder

from pathlib import Path

pd.set_option("display.max_columns", 50)

In [2]:
DATA_DIR = Path("/lustre09/project/6081099/reem2005/DATASET/PROCESSED")
CLEAN_INPUT = DATA_DIR / "realworld_clean.pkl"

final_df = pd.read_pickle(CLEAN_INPUT)

print("Shape:", final_df.shape)
assert "segment_id" in final_df.columns, "segment_id missing — did you load the wrong file?"
print("✅ segment_id present.")

Shape: (72309356, 10)
✅ segment_id present.


In [3]:
FS = 50
GRAVITY_CUTOFF = 0.3
FILTER_ORDER = 4

WINDOW_SIZE = 100       # 2 seconds @ 50Hz — same as the feature-extraction notebook
STEP_SIZE = 50          # 50% overlap

MERGE_TOLERANCE_MS = 10

CHANNEL_ORDER = ["body_x", "body_y", "body_z",
                  "gyr_x", "gyr_y", "gyr_z",
                  "mag_x", "mag_y", "mag_z"]

print(f"Window size: {WINDOW_SIZE} samples ({WINDOW_SIZE/FS:.1f}s) | Step: {STEP_SIZE} | Channels: {len(CHANNEL_ORDER)}")

Window size: 100 samples (2.0s) | Step: 50 | Channels: 9


In [4]:
def gravity_filter(signal: np.ndarray) -> np.ndarray:
    """Low-pass Butterworth filter to isolate gravity; returns the
    body-motion component (signal minus gravity). Must be called on a
    single continuous segment only."""
    nyquist = FS / 2
    b, a = butter(FILTER_ORDER, GRAVITY_CUTOFF / nyquist, btype="low")
    gravity = filtfilt(b, a, signal)
    return signal - gravity

In [5]:
def create_raw_windows(merged: pd.DataFrame, window_size: int = WINDOW_SIZE, step_size: int = STEP_SIZE):
    """Yields raw (window_size, n_channels) numpy arrays by row position.
    Safe here ONLY because the caller guarantees `merged` is a single
    continuous segment."""
    values = merged[CHANNEL_ORDER].to_numpy()
    for start in range(0, len(values) - window_size + 1, step_size):
        yield values[start:start + window_size]

## Pre-index gyr/mag for Fast Lookup

Grouping once up front (instead of filtering the full ~24M-row tables
inside the segment loop) is what makes this loop finish in minutes
instead of hours — the same fix applied in the feature-extraction notebook.

In [6]:
acc_df = final_df[final_df["sensor"] == "acc"].copy()
gyr_df = final_df[final_df["sensor"] == "gyr"].copy()
mag_df = final_df[final_df["sensor"] == "mag"].copy()

print("Pre-indexing gyr and mag by (participant, activity, position)...")
t0 = time.time()

gyr_lookup = {
    key: g.sort_values("attr_time")
    for key, g in gyr_df.groupby(["participant", "activity", "position"], observed=True)
}
mag_lookup = {
    key: g.sort_values("attr_time")
    for key, g in mag_df.groupby(["participant", "activity", "position"], observed=True)
}

print(f"gyr_lookup: {len(gyr_lookup)} groups | mag_lookup: {len(mag_lookup)} groups")
print(f"Pre-indexing took {time.time() - t0:.1f} seconds")

Pre-indexing gyr and mag by (participant, activity, position)...
gyr_lookup: 838 groups | mag_lookup: 838 groups
Pre-indexing took 12.4 seconds


## Main Windowing Loop

For every continuous segment: gravity-filter the accelerometer signal,
synchronize acc/gyr/mag via `merge_asof`, then slide raw windows across
the segment. Progress is printed periodically with an ETA.

In [7]:
X_list = []
y_list = []
groups_list = []

n_segments_processed = 0
n_segments_skipped_short = 0
n_segments_skipped_missing = 0

group_cols = ["participant", "activity", "position", "segment_id"]
MIN_SEGMENT_LENGTH = WINDOW_SIZE

TOTAL_SEGMENTS = final_df.groupby(group_cols, observed=True).ngroups
PRINT_EVERY = 500
loop_start = time.time()

print(f"Total segments to iterate: {TOTAL_SEGMENTS}\n")

for i, ((participant, activity, position, segment_id), acc_segment) in enumerate(
    acc_df.groupby(group_cols, observed=True), start=1
):
    acc_segment = acc_segment.sort_values("attr_time").copy()

    if len(acc_segment) < MIN_SEGMENT_LENGTH:
        n_segments_skipped_short += 1
    else:
        t_min, t_max = acc_segment["attr_time"].min(), acc_segment["attr_time"].max()
        pad = MERGE_TOLERANCE_MS
        key = (participant, activity, position)
        gyr_full = gyr_lookup.get(key)
        mag_full = mag_lookup.get(key)

        if gyr_full is None or mag_full is None:
            n_segments_skipped_missing += 1
        else:
            gyr_segment = gyr_full[gyr_full["attr_time"].between(t_min - pad, t_max + pad)]
            mag_segment = mag_full[mag_full["attr_time"].between(t_min - pad, t_max + pad)]

            if len(gyr_segment) < MIN_SEGMENT_LENGTH or len(mag_segment) < MIN_SEGMENT_LENGTH:
                n_segments_skipped_missing += 1
            else:
                for axis in ["attr_x", "attr_y", "attr_z"]:
                    acc_segment[f"body_{axis[-1]}"] = gravity_filter(acc_segment[axis].to_numpy())

                acc_processed = acc_segment[["attr_time", "body_x", "body_y", "body_z"]]
                gyr_processed = gyr_segment.rename(
                    columns={"attr_x": "gyr_x", "attr_y": "gyr_y", "attr_z": "gyr_z"}
                )[["attr_time", "gyr_x", "gyr_y", "gyr_z"]]
                mag_processed = mag_segment.rename(
                    columns={"attr_x": "mag_x", "attr_y": "mag_y", "attr_z": "mag_z"}
                )[["attr_time", "mag_x", "mag_y", "mag_z"]]

                merged = pd.merge_asof(
                    acc_processed, gyr_processed, on="attr_time",
                    direction="nearest", tolerance=MERGE_TOLERANCE_MS
                )
                merged = pd.merge_asof(
                    merged, mag_processed, on="attr_time",
                    direction="nearest", tolerance=MERGE_TOLERANCE_MS
                )
                merged = merged.dropna().reset_index(drop=True)

                if len(merged) < WINDOW_SIZE:
                    n_segments_skipped_short += 1
                else:
                    for window in create_raw_windows(merged, WINDOW_SIZE, STEP_SIZE):
                        X_list.append(window)
                        y_list.append(activity)
                        groups_list.append(participant)

                    n_segments_processed += 1

    if i % PRINT_EVERY == 0 or i == TOTAL_SEGMENTS:
        elapsed = time.time() - loop_start
        rate = i / elapsed
        remaining = (TOTAL_SEGMENTS - i) / rate if rate > 0 else float("inf")
        print(
            f"[{i}/{TOTAL_SEGMENTS}] "
            f"processed={n_segments_processed} skipped_short={n_segments_skipped_short} "
            f"skipped_missing={n_segments_skipped_missing} | "
            f"elapsed={elapsed/60:.1f} min | ~{remaining/60:.1f} min remaining | "
            f"windows_so_far={len(X_list)}"
        )

print(f"\nDone in {(time.time() - loop_start)/60:.1f} minutes")
print(f"Segments processed          : {n_segments_processed}")
print(f"Segments skipped (too short): {n_segments_skipped_short}")
print(f"Segments skipped (missing sensor data): {n_segments_skipped_missing}")
print(f"Total windows collected     : {len(X_list)}")

Total segments to iterate: 18876

[500/18876] processed=467 skipped_short=30 skipped_missing=3 | elapsed=0.1 min | ~5.4 min remaining | windows_so_far=11742
[1000/18876] processed=929 skipped_short=64 skipped_missing=7 | elapsed=0.2 min | ~3.4 min remaining | windows_so_far=25694
[1500/18876] processed=1416 skipped_short=76 skipped_missing=8 | elapsed=0.2 min | ~2.8 min remaining | windows_so_far=39246
[2000/18876] processed=1892 skipped_short=100 skipped_missing=8 | elapsed=0.3 min | ~2.4 min remaining | windows_so_far=52318
[2500/18876] processed=2360 skipped_short=132 skipped_missing=8 | elapsed=0.3 min | ~2.2 min remaining | windows_so_far=63901
[3000/18876] processed=2828 skipped_short=163 skipped_missing=9 | elapsed=0.4 min | ~2.0 min remaining | windows_so_far=77277
[3500/18876] processed=3315 skipped_short=175 skipped_missing=10 | elapsed=0.4 min | ~1.9 min remaining | windows_so_far=89889
[4000/18876] processed=3795 skipped_short=192 skipped_missing=13 | elapsed=0.5 min | ~1.7

## Stack into Arrays + Encode Labels

- `X_windows`: shape `(n_windows, window_size, n_channels)`
- `y_windows`: integer-encoded activity labels
- `groups`: integer-encoded participant IDs (for LOSO splitting)

Label encoders are saved alongside the arrays so the original activity
names and participant IDs can always be recovered.

In [8]:
X_windows = np.stack(X_list).astype("float32")

activity_encoder = LabelEncoder()
y_windows = activity_encoder.fit_transform(y_list).astype("int64")

participant_encoder = LabelEncoder()
groups_encoded = participant_encoder.fit_transform(groups_list).astype("int64")

print("X_windows shape:", X_windows.shape)
print("y_windows shape:", y_windows.shape)
print("groups shape    :", groups_encoded.shape)

print("\nActivity classes (index -> name):")
for idx, name in enumerate(activity_encoder.classes_):
    print(f"  {idx}: {name}")

print("\nParticipant classes (index -> name):")
for idx, name in enumerate(participant_encoder.classes_):
    print(f"  {idx}: {name}")

X_windows shape: (435295, 100, 9)
y_windows shape: (435295,)
groups shape    : (435295,)

Activity classes (index -> name):
  0: climbingdown
  1: climbingup
  2: jumping
  3: lying
  4: running
  5: sitting
  6: standing
  7: walking

Participant classes (index -> name):
  0: proband1
  1: proband10
  2: proband11
  3: proband12
  4: proband13
  5: proband14
  6: proband15
  7: proband2
  8: proband3
  9: proband4
  10: proband5
  11: proband6
  12: proband7
  13: proband8
  14: proband9


In [9]:
print("Any NaNs in X_windows:", np.isnan(X_windows).sum())
print("\nWindows per class:")
unique, counts = np.unique(y_windows, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {activity_encoder.classes_[u]}: {c}")

print("\nWindows per participant:")
unique, counts = np.unique(groups_encoded, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {participant_encoder.classes_[u]}: {c}")

Any NaNs in X_windows: 0

Windows per class:
  climbingdown: 48295
  climbingup: 59670
  jumping: 9259
  lying: 62692
  running: 68882
  sitting: 62525
  standing: 61542
  walking: 62430

Windows per participant:
  proband1: 28548
  proband10: 27340
  proband11: 28642
  proband12: 27257
  proband13: 28232
  proband14: 27576
  proband15: 28567
  proband2: 25608
  proband3: 29629
  proband4: 33200
  proband5: 31950
  proband6: 28113
  proband7: 29588
  proband8: 31779
  proband9: 29266


In [10]:
np.save(DATA_DIR / "X_windows.npy", X_windows)
np.save(DATA_DIR / "y_windows.npy", y_windows)
np.save(DATA_DIR / "groups.npy", groups_encoded)

np.save(DATA_DIR / "activity_classes.npy", activity_encoder.classes_)
np.save(DATA_DIR / "participant_classes.npy", participant_encoder.classes_)

print("Saved:")
for f in ["X_windows.npy", "y_windows.npy", "groups.npy", "activity_classes.npy", "participant_classes.npy"]:
    path = DATA_DIR / f
    print(f"  {f} — {round(path.stat().st_size / 1e6, 2)} MB")

Saved:
  X_windows.npy — 1567.06 MB
  y_windows.npy — 3.48 MB
  groups.npy — 3.48 MB
  activity_classes.npy — 0.0 MB
  participant_classes.npy — 0.0 MB


## Summary

- Loaded segment-aware cleaned RealWorld data
- Gravity filtering, sensor synchronization, and windowing performed
  within each continuous segment — no window crosses a recording gap
- Produced raw 9-channel windows (`body_x/y/z`, `gyr_x/y/z`, `mag_x/y/z`),
  each 100 timesteps (2s @ 50Hz)
- Saved `X_windows.npy`, `y_windows.npy`, `groups.npy` in the same format
  used by the earlier student-dataset model-comparison notebook

**Next notebook:** re-run the DNN / CNN1D / RNN / DeepConvLSTM / TinyHAR
comparison on this new RealWorld dataset, and compare against the
Random Forest / classical ML results from the feature-extraction path.